In [1]:
# Instalar dependencias que no vienen en Colab
!pip install -q pywavelets yfinance

import os
import csv
import time
import json
import argparse
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import pywt
import yfinance as yf
from scipy import signal
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.cm as cm

warnings.filterwarnings("ignore")


In [2]:
# ============================================================================
#  CONFIGURACION DE RUTAS  ---  ejecutar PRIMERO
#  USE_DRIVE = True  -> intenta usar Google Drive (persiste entre sesiones).
#                       Si el montaje falla, cae automaticamente a /content.
#  USE_DRIVE = False -> todo vive en /content (se borra al cerrar la sesion).
# ============================================================================
import os

USE_DRIVE = True   # <- ponelo en False para saltear Drive directamente

BASE_DIR = "/content"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        BASE_DIR = "/content/drive/MyDrive/VpC2_FinalProject"
        print("[OK] Drive montado.")
    except Exception as e:
        print(f"[!] No se pudo montar Drive ({e}). Sigo en /content (no persiste).")
        BASE_DIR = "/content"

os.makedirs(BASE_DIR, exist_ok=True)

STFT_DIR    = os.path.join(BASE_DIR, "dataset_stft")
WAVELET_DIR = os.path.join(BASE_DIR, "dataset_wavelet")

print("BASE_DIR    :", BASE_DIR)
print("STFT_DIR    :", STFT_DIR)
print("WAVELET_DIR :", WAVELET_DIR)


Mounted at /content/drive
[OK] Drive montado.
BASE_DIR    : /content/drive/MyDrive/VpC2_FinalProject
STFT_DIR    : /content/drive/MyDrive/VpC2_FinalProject/dataset_stft
WAVELET_DIR : /content/drive/MyDrive/VpC2_FinalProject/dataset_wavelet


## Generación de dataset

In [3]:
#!/usr/bin/env python3
"""
=============================================================================
 GENERADOR DE DATASETS DE ESPECTROGRAMAS FINANCIEROS
 Multi-activo: crypto, commodities, stocks, FX
 Horizontes de etiquetado: 1d, 5d, 10d, 20d, 30d
 Etiquetas: BUY / HOLD / SELL  (umbrales adaptativos por ATR)
 2026 reservado como test set

 Genera DOS datasets con la MISMA estructura y etiquetas:
   dataset_stft/    — imágenes basadas en Short-Time Fourier Transform
   dataset_wavelet/ — imágenes basadas en Continuous Wavelet Transform (Morlet)

 Estructura de cada dataset:
   images/
     train/   BTC-USD_00060.png ...
     test/    ...
   labels.csv
     sample_id, filename, ticker, asset_class, date, split, atr,
     ret_1d,  ret_5d,  ret_10d,  ret_20d,  ret_30d,
     label_1d, label_5d, label_10d, label_20d, label_30d
=============================================================================
"""



# ═════════════════════════════════════════════════════════════════════════════
# UNIVERSO DE ACTIVOS
# ═════════════════════════════════════════════════════════════════════════════

ASSET_UNIVERSE = {
    "crypto": [
        # "BTC-USD", "ETH-USD", "SOL-USD",
        "BTC-USD",
    ],
    "stocks": [
    #    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA","SPY",
         "AAPL",
    ],
    "commodities": [
        "GC=F", "SI=F",
    ],
    "fx": [
        "EURUSD=X", "GBPUSD=X",
    ],
}

# ═════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ═════════════════════════════════════════════════════════════════════════════

HORIZONS    = [3, 7, 15, 30]
WINDOW_DAYS = 60        # días de historia para cada muestra
IMG_SIZE    = 64        # píxeles de salida
CMAP        = "magma"
PAD_LENGTH  = 16        # días de padding por reflexión
START_DATE  = "2022-01-01"
TRAIN_END   = "2025-12-31"
TEST_START  = "2026-01-01"
MIN_HISTORY = 200       # mínimo de días para procesar un activo

# Configuración STFT
STFT_CFG = {
    "stft_window":  32,   # tamaño de ventana Hann
    "stft_overlap": 28,   # solapamiento entre ventanas
    "nfft":         64,   # resolución FFT (zero-padding)
}

# Configuración Wavelet de Morlet continua
# cmor{B}-{C}: B = bandwidth (ancho de banda), C = frecuencia central
# cmor1.5-1.0 es el estándar en análisis financiero
# scales: cada valor s representa un ciclo de aproximadamente s días
WAVELET_CFG = {
    "wavelet": "cmor1.5-1.0",
    "scales":  list(range(2, 31)),  # ciclos de 2 a 30 días
}

ATR_CFG = {
    "period":           14,
    "buy_multiplier":  0.7,   # bajado de 1.3 -> banda HOLD mas angosta, mas BUY/SELL
    "sell_multiplier": 0.7,   # bajado de 1.3 para balancear clases
}


# ═════════════════════════════════════════════════════════════════════════════
# PREPROCESAMIENTO DE SEÑAL
# ═════════════════════════════════════════════════════════════════════════════

def log_returns(close: np.ndarray) -> np.ndarray:
    """r_t = log(P_t / P_{t-1})  —  serie estacionaria y aditiva."""
    return np.diff(np.log(close))


def zscore(x: np.ndarray) -> np.ndarray:
    """Normalización Z. Media 0, std 1. Hace comparables activos distintos."""
    std = x.std()
    return (x - x.mean()) / std if std > 1e-10 else np.zeros_like(x)


def compute_atr(high: np.ndarray, low: np.ndarray,
                close: np.ndarray, period: int = 14) -> np.ndarray:
    """
    ATR como porcentaje del precio (EMA).
    Umbral dinámico: refleja la volatilidad real del activo en ese momento.
    Lo escalamos por sqrt(horizon) al etiquetar porque la volatilidad
    crece con la raíz del tiempo.
    """
    tr = np.maximum(
        high[1:] - low[1:],
        np.maximum(
            np.abs(high[1:] - close[:-1]),
            np.abs(low[1:] - close[:-1]),
        )
    )
    atr_pct = tr / close[:-1]
    atr = np.full_like(atr_pct, np.nan)
    atr[period - 1] = atr_pct[:period].mean()
    alpha = 2.0 / (period + 1)
    for i in range(period, len(atr_pct)):
        atr[i] = alpha * atr_pct[i] + (1 - alpha) * atr[i - 1]
    return atr


def make_label(future_return: float, threshold: float) -> str:
    """
    BUY  →  retorno > +threshold   (señal alcista fuerte)
    SELL →  retorno < -threshold   (señal bajista fuerte)
    HOLD →  cualquier cosa en el medio
    """
    if   future_return >  threshold: return "BUY"
    elif future_return < -threshold: return "SELL"
    else:                            return "HOLD"


# ═════════════════════════════════════════════════════════════════════════════
# GENERACIÓN DE IMÁGENES
# ═════════════════════════════════════════════════════════════════════════════

def make_stft_image(window: np.ndarray) -> np.ndarray:
    """
    STFT: aplica FFT sobre ventanas deslizantes con forma de Hann.
    Resolución tiempo-frecuencia fija para todas las frecuencias.
    Retorna matriz 2D en dB: (n_frecuencias, n_ventanas_temporales)
    """
    padded = np.pad(window, (0, PAD_LENGTH), mode="reflect")
    _, _, Sxx = signal.spectrogram(
        padded,
        fs=1.0,
        window=signal.windows.hann(STFT_CFG["stft_window"]),
        noverlap=STFT_CFG["stft_overlap"],
        nfft=STFT_CFG["nfft"],
        scaling="density",
    )
    # Recortar las columnas del padding
    n_cols_orig = (WINDOW_DAYS - STFT_CFG["stft_window"]) // (
        STFT_CFG["stft_window"] - STFT_CFG["stft_overlap"]
    ) + 1
    return 10 * np.log10(Sxx[:, :n_cols_orig] + 1e-12)


def make_wavelet_image(window: np.ndarray) -> np.ndarray:
    """
    CWT con Wavelet de Morlet compleja (cmor1.5-1.0).
    Resolución adaptativa: ventanas cortas para ciclos cortos,
    ventanas largas para ciclos largos.
    Retorna matriz 2D en dB: (n_scales, n_tiempo)
    Los scales cubren ciclos de 2 a 30 días.
    """
    padded = np.pad(window, (0, PAD_LENGTH), mode="reflect")
    scales = np.array(WAVELET_CFG["scales"], dtype=float)
    coeffs, _ = pywt.cwt(
        padded, scales, WAVELET_CFG["wavelet"], sampling_period=1.0
    )
    power = np.abs(coeffs) ** 2
    # Recortar padding: las primeras WINDOW_DAYS columnas son señal real
    power = power[:, :WINDOW_DAYS]
    return 10 * np.log10(power + 1e-12)


def save_png(matrix: np.ndarray, filepath: str):
    """
    Convierte una matriz 2D a imagen PNG 64×64 usando PIL (sin matplotlib).
    Normaliza a [0,1], aplica colormap magma, flip vertical
    para que frecuencias bajas queden abajo.
    """
    vmin, vmax = matrix.min(), matrix.max()
    norm  = (matrix - vmin) / (vmax - vmin + 1e-10)
    rgba  = cm.get_cmap(CMAP)(norm)
    rgb   = (rgba[:, :, :3] * 255).astype(np.uint8)
    rgb   = rgb[::-1]  # frecuencias bajas abajo
    Image.fromarray(rgb).resize(
        (IMG_SIZE, IMG_SIZE), Image.BILINEAR
    ).save(filepath, format="PNG")


# ═════════════════════════════════════════════════════════════════════════════
# DESCARGA
# ═════════════════════════════════════════════════════════════════════════════

def download(ticker: str, start: str) -> pd.DataFrame | None:
    try:
        df = yf.download(
            ticker, start=start, interval="1d",
            progress=False, auto_adjust=True,
        )
        if df is None or len(df) < MIN_HISTORY:
            print(f"  ⚠  {ticker}: {len(df) if df is not None else 0} registros")
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        return df[["Close", "High", "Low"]].dropna()
    except Exception as e:
        print(f"  ✗  {ticker}: {e}")
        return None


# ═════════════════════════════════════════════════════════════════════════════
# PROCESAMIENTO DE UN ACTIVO
# ═════════════════════════════════════════════════════════════════════════════

def process_asset(ticker: str, asset_class: str, df: pd.DataFrame,
                  img_fn,             # make_stft_image o make_wavelet_image
                  img_dir_train: str,
                  img_dir_test: str) -> list[dict]:
    """
    Genera imágenes y registros de etiquetas para un activo.
    img_fn es la función de transformación: STFT o Wavelet.
    La misma lógica sirve para los dos datasets.
    """
    close  = df["Close"].values.astype(float)
    high   = df["High"].values.astype(float)
    low    = df["Low"].values.astype(float)
    dates  = df.index

    ret      = log_returns(close)
    ret_norm = zscore(ret)
    atr      = compute_atr(high, low, close, ATR_CFG["period"])

    max_horizon = max(HORIZONS)
    records     = []

    for i in range(WINDOW_DAYS, len(ret_norm) - max_horizon):

        # Validar ATR
        atr_val = atr[i] if i < len(atr) and not np.isnan(atr[i]) else None
        if not atr_val or atr_val < 1e-8:
            continue

        sample_date = dates[i + 1]  # +1 por offset de log_returns
        split       = "test" if sample_date >= pd.Timestamp(TEST_START) else "train"
        img_dir     = img_dir_test if split == "test" else img_dir_train
        sample_id   = f"{ticker}_{i:05d}"
        fname       = f"{sample_id}.png"

        # Generar y guardar imagen
        window   = ret_norm[i - WINDOW_DAYS:i]
        matrix   = img_fn(window)
        save_png(matrix, os.path.join(img_dir, fname))

        # Calcular retornos y etiquetas para todos los horizontes
        record = {
            "sample_id":   sample_id,
            "filename":    fname,
            "ticker":      ticker,
            "asset_class": asset_class,
            "date":        str(sample_date.date()),
            "split":       split,
            "atr":         round(float(atr_val), 6),
        }
        for h in HORIZONS:
            future_ret       = float(np.sum(ret[i:i + h]))
            threshold        = atr_val * ATR_CFG["buy_multiplier"] * np.sqrt(h)
            record[f"ret_{h}d"]   = round(future_ret, 6)
            record[f"label_{h}d"] = make_label(future_ret, threshold)

        records.append(record)

        if len(records) % 200 == 0:
            print(f"     ... {len(records)} muestras", end="\r")

    return records


# ═════════════════════════════════════════════════════════════════════════════
# PIPELINE DE GENERACIÓN
# ═════════════════════════════════════════════════════════════════════════════

CSV_FIELDS = (
    ["sample_id", "filename", "ticker", "asset_class", "date", "split", "atr"]
    + [f"ret_{h}d"   for h in HORIZONS]
    + [f"label_{h}d" for h in HORIZONS]
)


def generate(output_dir: str, img_fn, name: str,
             asset_classes: list, start_date: str):
    """
    Genera un dataset completo con la función de imagen indicada.
    name: "STFT" o "Wavelet"  (solo para logging)
    img_fn: make_stft_image o make_wavelet_image
    """
    img_train = os.path.join(output_dir, "images", "train")
    img_test  = os.path.join(output_dir, "images", "test")
    os.makedirs(img_train, exist_ok=True)
    os.makedirs(img_test,  exist_ok=True)

    csv_path = os.path.join(output_dir, "labels.csv")
    if os.path.exists(csv_path):
        os.remove(csv_path)

    print("\n" + "═" * 65)
    print(f" DATASET {name}")
    print(f" Output  : {output_dir}")
    print(f" Ventana : {WINDOW_DAYS} días")
    print(f" Train   : {start_date} → {TRAIN_END}")
    print(f" Test    : {TEST_START} → hoy")
    print("═" * 65)

    total        = 0
    label_counts = {h: {"BUY": 0, "HOLD": 0, "SELL": 0}
                    for h in HORIZONS}

    for asset_class in asset_classes:
        tickers = ASSET_UNIVERSE.get(asset_class, [])
        print(f"\n── {asset_class.upper()} ({len(tickers)} activos) ──")

        for ticker in tickers:
            print(f"  📥 {ticker} ...", end=" ", flush=True)
            df = download(ticker, start_date)
            if df is None:
                continue
            print(f"{len(df)} días  "
                  f"{df.index[0].date()} → {df.index[-1].date()}")

            t0      = time.time()
            records = process_asset(
                ticker, asset_class, df,
                img_fn, img_train, img_test,
            )
            elapsed = time.time() - t0

            if not records:
                print("     ⚠  Sin muestras generadas")
                continue

            # Escribir al CSV de forma incremental
            exists = os.path.exists(csv_path)
            with open(csv_path, "a", newline="", encoding="utf-8") as fh:
                writer = csv.DictWriter(fh, fieldnames=CSV_FIELDS)
                if not exists:
                    writer.writeheader()
                writer.writerows(records)

            total += len(records)
            for r in records:
                for h in HORIZONS:
                    label_counts[h][r[f"label_{h}d"]] += 1

            speed = len(records) / max(elapsed, 0.01)
            print(f"     ✅ {len(records)} muestras  "
                  f"{elapsed:.1f}s  ({speed:.0f} img/s)")

    # ── Resumen ───────────────────────────────────────────────────────────
    print(f"\n{'═'*65}")
    print(f" {name} — {total} muestras totales")
    print(f"{'─'*65}")

    df_csv = pd.read_csv(csv_path) if os.path.exists(csv_path) else None
    if df_csv is not None:
        tr = len(df_csv[df_csv["split"] == "train"])
        te = len(df_csv[df_csv["split"] == "test"])
        print(f" Train: {tr}   Test: {te}")

    for h in HORIZONS:
        lc = label_counts[h]
        th = max(sum(lc.values()), 1)
        line = f"  {h:2d}d: "
        for lbl in ["BUY", "HOLD", "SELL"]:
            line += f" {lbl} {lc[lbl]:5d} ({lc[lbl]/th*100:.1f}%)"
        print(line)

    # ── Config ────────────────────────────────────────────────────────────
    cfg = {
        "dataset":        name,
        "generated_at":   datetime.now().isoformat(),
        "start_date":     start_date,
        "train_end":      TRAIN_END,
        "test_start":     TEST_START,
        "window_days":    WINDOW_DAYS,
        "img_size":       IMG_SIZE,
        "horizons":       HORIZONS,
        "atr_config":     ATR_CFG,
        "asset_classes":  asset_classes,
        "asset_universe": {k: ASSET_UNIVERSE[k] for k in asset_classes},
        "transform_config": STFT_CFG if name == "STFT" else WAVELET_CFG,
    }
    with open(os.path.join(output_dir, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    print(f"\n  labels.csv  →  {csv_path}")
    print(f"  config.json →  {os.path.join(output_dir, 'config.json')}")
    return total


# ═════════════════════════════════════════════════════════════════════════════
# VERIFICACIÓN
# ═════════════════════════════════════════════════════════════════════════════

def verify(output_dir: str):
    csv_path = os.path.join(output_dir, "labels.csv")
    if not os.path.exists(csv_path):
        print(f"✗ No se encontró labels.csv en {output_dir}")
        return

    df = pd.read_csv(csv_path)
    print(f"\n🔍 Verificando {output_dir}")
    print(f"   labels.csv: {len(df)} filas  ·  {len(df.columns)} columnas")

    for split in ["train", "test"]:
        img_path = os.path.join(output_dir, "images", split)
        n_imgs   = len(list(os.scandir(img_path))) if os.path.exists(img_path) else 0
        n_csv    = len(df[df["split"] == split])
        ok       = "✅" if n_imgs == n_csv else "⚠️ "
        print(f"   {ok} {split:5s}: {n_imgs} imágenes  {n_csv} en CSV")

    print(f"\n   Distribución de etiquetas:")
    for h in HORIZONS:
        col = f"label_{h}d"
        if col not in df.columns:
            continue
        counts = df[col].value_counts()
        line   = f"   {h:2d}d:"
        for lbl in ["BUY", "HOLD", "SELL"]:
            n = counts.get(lbl, 0)
            line += f"  {lbl} {n:5d} ({n/len(df)*100:.1f}%)"
        print(line)

In [4]:
# ============================================================================
#  EJECUCION  ---  genera el dataset SOLO si todavia no existe.
#  Si ya esta en Drive, salta la generacion y pasa directo a entrenar.
# ============================================================================
import time

classes = ["crypto", "stocks", "commodities", "fx"]

def _dataset_listo(d):
    return os.path.exists(os.path.join(d, "labels.csv"))

t0 = time.time()

if _dataset_listo(STFT_DIR):
    print("[skip] STFT ya existe en", STFT_DIR)
else:
    generate(output_dir=STFT_DIR, img_fn=make_stft_image, name="STFT",
             asset_classes=classes, start_date=START_DATE)

if _dataset_listo(WAVELET_DIR):
    print("[skip] Wavelet ya existe en", WAVELET_DIR)
else:
    generate(output_dir=WAVELET_DIR, img_fn=make_wavelet_image, name="Wavelet",
             asset_classes=classes, start_date=START_DATE)

print("Tiempo total: {:.1f} min".format((time.time() - t0) / 60))

verify(STFT_DIR)
verify(WAVELET_DIR)


[skip] STFT ya existe en /content/drive/MyDrive/VpC2_FinalProject/dataset_stft
[skip] Wavelet ya existe en /content/drive/MyDrive/VpC2_FinalProject/dataset_wavelet
Tiempo total: 0.0 min

🔍 Verificando /content/drive/MyDrive/VpC2_FinalProject/dataset_stft
   labels.csv: 6709 filas  ·  15 columnas
   ✅ train: 6186 imágenes  6186 en CSV
   ✅ test : 523 imágenes  523 en CSV

   Distribución de etiquetas:
    3d:  BUY  1083 (16.1%)  HOLD  4720 (70.4%)  SELL   906 (13.5%)
    7d:  BUY  1203 (17.9%)  HOLD  4614 (68.8%)  SELL   892 (13.3%)
   15d:  BUY  1289 (19.2%)  HOLD  4555 (67.9%)  SELL   865 (12.9%)
   30d:  BUY  1436 (21.4%)  HOLD  4452 (66.4%)  SELL   821 (12.2%)

🔍 Verificando /content/drive/MyDrive/VpC2_FinalProject/dataset_wavelet
   labels.csv: 6709 filas  ·  15 columnas
   ✅ train: 6186 imágenes  6186 en CSV
   ✅ test : 523 imágenes  523 en CSV

   Distribución de etiquetas:
    3d:  BUY  1083 (16.1%)  HOLD  4720 (70.4%)  SELL   906 (13.5%)
    7d:  BUY  1203 (17.9%)  HOLD  4614 (

## Entrenamiento de modelo

In [5]:
!pip install torch torchvision scikit-learn matplotlib seaborn

In [6]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

In [7]:

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Usando: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")



IMG_SIZE     = 64        # Debe coincidir con el tamano generado antes
EPOCHS       = 25
LR           = 3e-4      # Learning rate inicial
LR_PATIENCE  = 3         # baja LR mas rapido al estancarse val
EARLY_STOP   = 6         # corta antes -> evita memorizar el train
WEIGHT_DECAY = 3e-4      # mas regularizacion L2 contra overfitting
DROPOUT      = 0.55      # dropout alto en la cabeza de la CNN

✅ Usando: cuda
   GPU: Tesla T4


In [8]:
# ============================================================================
#  CONFIG DE ENTRENAMIENTO + Dataset + builder de DataLoaders
#  Todo parametrizado para poder comparar modelos bajo MISMAS condiciones.
# ============================================================================
import pandas as pd
from PIL import Image as PILImage

HORIZON    = 7                       # 3, 7, 15, 30
CLASSES    = ["BUY", "HOLD", "SELL"]
BATCH_SIZE = 64
VAL_CUTOFF = "2025-02-01"            # corte cronologico train / val

# Transforms (mismos para todos los experimentos -> comparacion justa)
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=0, translate=(0.02, 0.0)),  # solo eje X (tiempo)
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


class SpectrogramDataset(Dataset):
    """Lee imagenes desde images/{split}/ y etiquetas desde labels.csv."""
    def __init__(self, df, img_dir, horizon, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.label_col = f"label_{horizon}d"
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = PILImage.open(os.path.join(self.img_dir, row["filename"])).convert("RGB")
        label = CLASSES.index(row[self.label_col])
        if self.transform:
            img = self.transform(img)
        return img, label

    def get_labels(self):
        return [CLASSES.index(lbl) for lbl in self.df[self.label_col]]


def build_loaders(dataset_dir, horizon=HORIZON, verbose=True):
    """Devuelve loaders + class_weight_tensor para un dataset dado.
    Split cronologico identico para STFT y Wavelet -> condiciones iguales."""
    csv_path = os.path.join(dataset_dir, "labels.csv")
    df_all   = pd.read_csv(csv_path, parse_dates=["date"])

    df_train_full = df_all[df_all["split"] == "train"].copy()
    df_test       = df_all[df_all["split"] == "test"].copy()
    df_train = df_train_full[df_train_full["date"] <  VAL_CUTOFF]
    df_val   = df_train_full[df_train_full["date"] >= VAL_CUTOFF]

    img_train_dir = os.path.join(dataset_dir, "images", "train")
    img_test_dir  = os.path.join(dataset_dir, "images", "test")

    train_ds = SpectrogramDataset(df_train, img_train_dir, horizon, train_tf)
    val_ds   = SpectrogramDataset(df_val,   img_train_dir, horizon, eval_tf)
    test_ds  = SpectrogramDataset(df_test,  img_test_dir,  horizon, eval_tf)

    # WeightedRandomSampler (desbalance)
    train_labels = train_ds.get_labels()
    class_counts = Counter(train_labels)
    total_train  = len(train_labels)
    cw = {c: total_train / max(class_counts.get(c, 0), 1) for c in range(len(CLASSES))}
    sample_weights = [cw[lbl] for lbl in train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    # Pesos por clase para el loss
    _counts = torch.tensor([class_counts.get(c, 0) for c in range(len(CLASSES))],
                           dtype=torch.float)
    cwt = total_train / (len(CLASSES) * _counts.clamp(min=1))
    cwt = (cwt / cwt.sum() * len(CLASSES)).to(DEVICE)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    if verbose:
        print(f"[{os.path.basename(dataset_dir)}] horizonte {horizon}d  |  "
              f"train {len(train_ds)}  val {len(val_ds)}  test {len(test_ds)}")
        dist = {cls: class_counts.get(i, 0) for i, cls in enumerate(CLASSES)}
        print("   train dist:", dist,
              "   pesos loss:", {c: round(w, 2) for c, w in zip(CLASSES, cwt.tolist())})

    return {"train": train_loader, "val": val_loader, "test": test_loader,
            "class_weight": cwt, "test_ds": test_ds}


In [9]:
# ============================================================================
#  ARQUITECTURAS  ---  CNN custom y ResNet18 (transfer learning)
#  build_model("cnn" | "resnet") instancia cada una con las mismas clases.
# ============================================================================
class SpectrogramCNN(nn.Module):
    """CNN de 4 bloques convolucionales para espectrogramas 64x64."""
    def __init__(self, num_classes=3, dropout=None):
        super().__init__()
        dropout = DROPOUT if dropout is None else dropout
        def conv_block(in_ch, out_ch, pool=True, drop=0.2):
            layers = [
                nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ]
            if pool:
                layers.append(nn.MaxPool2d(2, 2))
            layers.append(nn.Dropout2d(drop))
            return nn.Sequential(*layers)
        self.features = nn.Sequential(
            conv_block(3,   32,  drop=0.1),
            conv_block(32,  64,  drop=0.15),
            conv_block(64,  128, drop=0.2),
            conv_block(128, 256, drop=0.25),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((2, 2)), nn.Flatten(),
            nn.Linear(1024, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, 64),   nn.ReLU(inplace=True), nn.Dropout(dropout / 2),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


class ResNetFinancial(nn.Module):
    """ResNet18 preentrenado, fine-tuning de las ultimas capas."""
    def __init__(self, num_classes=3, freeze_backbone=True):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        if freeze_backbone:
            for p in list(base.parameters())[:-10]:
                p.requires_grad = False
        in_f = base.fc.in_features
        base.fc = nn.Sequential(
            nn.Linear(in_f, 128), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(128, num_classes),
        )
        self.model = base
    def forward(self, x):
        return self.model(x)


def build_model(arch, num_classes=len(CLASSES)):
    if arch == "cnn":
        return SpectrogramCNN(num_classes=num_classes).to(DEVICE)
    elif arch == "resnet":
        return ResNetFinancial(num_classes=num_classes).to(DEVICE)
    raise ValueError(f"arch desconocida: {arch}")


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(1)
        total_loss += loss.item() * imgs.size(0)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels, all_probs


In [10]:
# ============================================================================
#  run_experiment  ---  entrena UN modelo bajo condiciones controladas
#  y devuelve sus metricas en test. Reusa la misma seed, epochs, loss, etc.
# ============================================================================
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report

def run_experiment(repr_name, arch, dataset_dir, horizon=HORIZON,
                   epochs=EPOCHS, verbose=True):
    """repr_name: 'STFT' | 'Wavelet'   arch: 'cnn' | 'resnet'"""
    tag = f"{arch.upper()}-{repr_name}"
    print(f"\n{'='*70}\n  EXPERIMENTO: {tag}\n{'='*70}")

    # Reproducibilidad por experimento (misma seed -> comparacion justa)
    torch.manual_seed(SEED); np.random.seed(SEED)

    loaders = build_loaders(dataset_dir, horizon)
    model = build_model(arch)
    criterion = nn.CrossEntropyLoss(weight=loaders["class_weight"], label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                     factor=0.5, patience=LR_PATIENCE)

    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
            "val_f1": [], "lr": []}
    best_f1, best_weights, patience = -1.0, None, 0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, loaders["train"], criterion, optimizer)
        vl_loss, vl_acc, vp, vl_lbl, _ = eval_epoch(model, loaders["val"], criterion)
        vf1 = f1_score(vl_lbl, vp, average="macro")
        scheduler.step(vl_loss)
        for k, v in zip(hist, [tr_loss, vl_loss, tr_acc, vl_acc, vf1,
                               optimizer.param_groups[0]["lr"]]):
            hist[k].append(v)
        if vf1 > best_f1:
            best_f1 = vf1
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
        if verbose:
            print(f"  ep {epoch:2d} | tr_loss {tr_loss:.3f} acc {tr_acc*100:5.1f}% | "
                  f"val_loss {vl_loss:.3f} acc {vl_acc*100:5.1f}% f1 {vf1:.3f}")
        if patience >= EARLY_STOP:
            print(f"  early stopping en epoch {epoch}")
            break

    model.load_state_dict(best_weights)

    # Evaluacion final en test
    te_loss, te_acc, preds, labels, probs = eval_epoch(model, loaders["test"], criterion)
    te_balacc = balanced_accuracy_score(labels, preds)
    te_f1     = f1_score(labels, preds, average="macro")
    n_params  = sum(p.numel() for p in model.parameters())

    print(f"\n  >> {tag}  TEST  acc {te_acc*100:.2f}%  "
          f"bal-acc {te_balacc*100:.2f}%  f1-macro {te_f1:.4f}")
    print(classification_report(labels, preds, target_names=CLASSES, digits=3))

    # Guardar pesos en Drive
    ckpt = os.path.join(BASE_DIR, f"model_{tag}.pth")
    torch.save(best_weights, ckpt)

    return {
        "modelo": arch.upper(), "repr": repr_name, "tag": tag,
        "test_acc": te_acc, "bal_acc": te_balacc, "f1_macro": te_f1,
        "val_f1_best": best_f1, "params": n_params,
        "history": hist, "preds": np.array(preds), "labels": np.array(labels),
        "probs": np.array(probs), "model": model, "test_ds": loaders["test_ds"],
        "ckpt": ckpt,
    }


In [11]:
# ============================================================================
#  COMPARACION DE MODELOS  ---  4 experimentos bajo las MISMAS condiciones
#  (2 arquitecturas x 2 representaciones). Misma seed, epochs, loss, split.
# ============================================================================
EXPERIMENTOS = [
    ("STFT",    "cnn",    STFT_DIR),
    ("Wavelet", "cnn",    WAVELET_DIR),
    ("STFT",    "resnet", STFT_DIR),
    ("Wavelet", "resnet", WAVELET_DIR),
]

resultados = []
for repr_name, arch, ddir in EXPERIMENTOS:
    res = run_experiment(repr_name, arch, ddir)
    resultados.append(res)

# Tabla comparativa
tabla = pd.DataFrame([{
    "Modelo": r["modelo"], "Repr": r["repr"],
    "Test Acc (%)": round(r["test_acc"] * 100, 2),
    "Bal-Acc (%)":  round(r["bal_acc"] * 100, 2),
    "F1-macro":     round(r["f1_macro"], 4),
    "Params":       f"{r['params']:,}",
} for r in resultados]).sort_values("F1-macro", ascending=False).reset_index(drop=True)

print("\n" + "=" * 70)
print("  TABLA COMPARATIVA  (ordenada por F1-macro)")
print("=" * 70)
print(tabla.to_string(index=False))

# Guardar tabla para el paper
tabla.to_csv(os.path.join(BASE_DIR, "tabla_comparativa.csv"), index=False)
print(f"\n[i] Tabla guardada en {os.path.join(BASE_DIR, 'tabla_comparativa.csv')}")

# El mejor modelo queda accesible para las celdas de analisis (Grad-CAM, etc.)
mejor = max(resultados, key=lambda r: r["f1_macro"])
model    = mejor["model"]
test_ds  = mejor["test_ds"]
history  = mejor["history"]
preds    = mejor["preds"]
labels   = mejor["labels"]
probs    = mejor["probs"]
print(f"\n[i] Mejor modelo: {mejor['tag']} (F1-macro {mejor['f1_macro']:.4f}) "
      f"-> usado en las celdas de analisis siguientes")



  EXPERIMENTO: CNN-STFT
[dataset_stft] horizonte 7d  |  train 4690  val 1496  test 523
   train dist: {'BUY': 831, 'HOLD': 3179, 'SELL': 680}    pesos loss: {'BUY': 1.21, 'HOLD': 0.32, 'SELL': 1.48}
  ep  1 | tr_loss 0.974 acc  34.9% | val_loss 1.408 acc   9.2% f1 0.056
  ep  2 | tr_loss 0.954 acc  36.6% | val_loss 1.361 acc  12.7% f1 0.133
  ep  3 | tr_loss 0.921 acc  41.5% | val_loss 1.408 acc  14.1% f1 0.143
  ep  4 | tr_loss 0.835 acc  49.1% | val_loss 1.455 acc  14.8% f1 0.149
  ep  5 | tr_loss 0.709 acc  55.4% | val_loss 1.528 acc  14.6% f1 0.147
  ep  6 | tr_loss 0.598 acc  61.3% | val_loss 1.738 acc  15.6% f1 0.156
  ep  7 | tr_loss 0.516 acc  64.0% | val_loss 1.813 acc  15.0% f1 0.156
  ep  8 | tr_loss 0.472 acc  65.2% | val_loss 1.611 acc  16.6% f1 0.163
  ep  9 | tr_loss 0.447 acc  67.2% | val_loss 1.700 acc  16.7% f1 0.164
  ep 10 | tr_loss 0.445 acc  65.8% | val_loss 1.658 acc  17.4% f1 0.172
  ep 11 | tr_loss 0.430 acc  65.5% | val_loss 1.659 acc  17.1% f1 0.171
  ep 12 

100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


  ep  1 | tr_loss 0.918 acc  42.2% | val_loss 1.501 acc  14.0% f1 0.146
  ep  2 | tr_loss 0.750 acc  52.8% | val_loss 1.399 acc  17.7% f1 0.166
  ep  3 | tr_loss 0.609 acc  59.0% | val_loss 1.552 acc  14.9% f1 0.149
  ep  4 | tr_loss 0.523 acc  64.0% | val_loss 1.548 acc  19.0% f1 0.182
  ep  5 | tr_loss 0.482 acc  66.0% | val_loss 1.506 acc  23.0% f1 0.229
  ep  6 | tr_loss 0.429 acc  70.4% | val_loss 1.458 acc  31.2% f1 0.280
  ep  7 | tr_loss 0.405 acc  73.5% | val_loss 1.448 acc  35.4% f1 0.293
  ep  8 | tr_loss 0.361 acc  77.4% | val_loss 1.472 acc  43.7% f1 0.314
  ep  9 | tr_loss 0.339 acc  80.9% | val_loss 1.504 acc  45.7% f1 0.325
  ep 10 | tr_loss 0.324 acc  82.7% | val_loss 1.543 acc  47.8% f1 0.327
  ep 11 | tr_loss 0.311 acc  84.5% | val_loss 1.531 acc  50.9% f1 0.327
  ep 12 | tr_loss 0.297 acc  86.0% | val_loss 1.532 acc  47.5% f1 0.324
  ep 13 | tr_loss 0.291 acc  87.5% | val_loss 1.583 acc  51.6% f1 0.332
  ep 14 | tr_loss 0.278 acc  87.9% | val_loss 1.569 acc  55.0% f

In [12]:
# ============================================================================
#  GRAFICO COMPARATIVO  ---  F1-macro y Balanced-Accuracy por modelo
# ============================================================================
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor("#0f0f23"); ax.set_facecolor("#1a1a2e")

x = np.arange(len(resultados))
w = 0.38
tags    = [r["tag"] for r in resultados]
f1s     = [r["f1_macro"] for r in resultados]
balaccs = [r["bal_acc"] for r in resultados]

ax.bar(x - w/2, f1s,     w, label="F1-macro",      color="#2ecc71")
ax.bar(x + w/2, balaccs, w, label="Balanced-Acc",  color="#3498db")
ax.axhline(1/len(CLASSES), color="#e74c3c", ls="--", lw=1,
           label=f"Azar ({1/len(CLASSES):.2f})")

ax.set_xticks(x); ax.set_xticklabels(tags, rotation=15, color="gray")
ax.set_title("Comparacion de modelos (test set 2026)", color="white",
             fontsize=13, fontweight="bold")
ax.set_ylim(0, 1); ax.tick_params(colors="gray")
ax.spines[:].set_color("#333355"); ax.grid(True, alpha=0.12, color="white", axis="y")
ax.legend(facecolor="#1a1a2e", labelcolor="white")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "comparacion_modelos.png"), dpi=130,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


In [13]:
# ============================================================================
#  BARRIDO DE HORIZONTES  ---  el mejor (arquitectura, representacion) a 7/15/30d
#  Horizontes largos suelen tener mas senal direccional y menos ruido.
# ============================================================================
mejor_arch = mejor["modelo"].lower()          # 'cnn' o 'resnet'
mejor_repr = mejor["repr"]
mejor_dir  = STFT_DIR if mejor_repr == "STFT" else WAVELET_DIR

HORIZONTES = [7, 15, 30]
barrido = []
for h in HORIZONTES:
    res_h = run_experiment(mejor_repr, mejor_arch, mejor_dir, horizon=h, verbose=False)
    barrido.append({
        "Horizonte (d)": h,
        "Test Acc (%)": round(res_h["test_acc"] * 100, 2),
        "Bal-Acc (%)":  round(res_h["bal_acc"] * 100, 2),
        "F1-macro":     round(res_h["f1_macro"], 4),
    })

tabla_h = pd.DataFrame(barrido)
print("\n" + "=" * 60)
print(f"  BARRIDO DE HORIZONTES  ({mejor['tag']})")
print("=" * 60)
print(tabla_h.to_string(index=False))
tabla_h.to_csv(os.path.join(BASE_DIR, "barrido_horizontes.csv"), index=False)

# Grafico
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor("#0f0f23"); ax.set_facecolor("#1a1a2e")
ax.plot(tabla_h["Horizonte (d)"], tabla_h["F1-macro"], "-o", color="#2ecc71", label="F1-macro")
ax.plot(tabla_h["Horizonte (d)"], tabla_h["Bal-Acc (%)"] / 100, "-o", color="#3498db", label="Bal-Acc")
ax.axhline(1/len(CLASSES), color="#e74c3c", ls="--", lw=1, label=f"Azar ({1/len(CLASSES):.2f})")
ax.set_xlabel("Horizonte (dias)", color="gray"); ax.set_ylim(0, 1)
ax.set_title(f"Efecto del horizonte  ({mejor['tag']})", color="white", fontweight="bold")
ax.tick_params(colors="gray"); ax.spines[:].set_color("#333355")
ax.grid(True, alpha=0.12, color="white"); ax.legend(facecolor="#1a1a2e", labelcolor="white")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "barrido_horizontes.png"), dpi=130,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()



  EXPERIMENTO: RESNET-STFT
[dataset_stft] horizonte 7d  |  train 4690  val 1496  test 523
   train dist: {'BUY': 831, 'HOLD': 3179, 'SELL': 680}    pesos loss: {'BUY': 1.21, 'HOLD': 0.32, 'SELL': 1.48}

  >> RESNET-STFT  TEST  acc 62.33%  bal-acc 37.43%  f1-macro 0.3692
              precision    recall  f1-score   support

         BUY      0.282     0.256     0.268        78
        HOLD      0.721     0.812     0.764       372
        SELL      0.121     0.055     0.075        73

    accuracy                          0.623       523
   macro avg      0.375     0.374     0.369       523
weighted avg      0.572     0.623     0.594       523


  EXPERIMENTO: RESNET-STFT
[dataset_stft] horizonte 15d  |  train 4690  val 1496  test 523
   train dist: {'BUY': 860, 'HOLD': 3136, 'SELL': 694}    pesos loss: {'BUY': 1.19, 'HOLD': 0.33, 'SELL': 1.48}
  early stopping en epoch 15

  >> RESNET-STFT  TEST  acc 51.63%  bal-acc 35.62%  f1-macro 0.3415
              precision    recall  f1-score  

In [14]:
# ── CELDA 19: Curvas de entrenamiento ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor("#0f0f23")
fig.suptitle("Historial de entrenamiento", color="white", fontsize=14, fontweight="bold")

ep = range(1, len(history["train_loss"]) + 1)
colors = {"train": "#3498db", "val": "#e74c3c"}

# Loss
axes[0].plot(ep, history["train_loss"], color=colors["train"], label="Train", linewidth=1.8)
axes[0].plot(ep, history["val_loss"],   color=colors["val"],   label="Val",   linewidth=1.8)
axes[0].set_title("Loss", color="white")
axes[0].set_xlabel("Epoch", color="gray")
axes[0].legend(facecolor="#1a1a2e", labelcolor="white")

# Accuracy
axes[1].plot(ep, [a * 100 for a in history["train_acc"]], color=colors["train"], label="Train", linewidth=1.8)
axes[1].plot(ep, [a * 100 for a in history["val_acc"]],   color=colors["val"],   label="Val",   linewidth=1.8)
axes[1].set_title("Accuracy (%)", color="white")
axes[1].set_xlabel("Epoch", color="gray")
axes[1].legend(facecolor="#1a1a2e", labelcolor="white")

# Learning rate
axes[2].semilogy(ep, history["val_f1"], color="#2ecc71", linewidth=1.8)
axes[2].set_title("F1 Macro", color="white")
axes[2].set_xlabel("Epoch", color="gray")

for ax in axes:
    ax.set_facecolor("#1a1a2e")
    ax.tick_params(colors="gray")
    ax.spines[:].set_color("#333355")
    ax.grid(True, alpha=0.12, color="white")

plt.tight_layout()
plt.show()

# ── CELDA 20: Evaluación en Test Set ─────────────────────────────────────────
# El mejor modelo y sus predicciones ya fueron calculados en la comparacion (celda 12).
test_acc = mejor["test_acc"]
preds  = np.array(mejor["preds"])
labels = np.array(mejor["labels"])
probs  = np.array(mejor["probs"])
probs  = np.array(probs)
preds  = np.array(preds)
labels = np.array(labels)

print(f"\n{'='*50}")
from sklearn.metrics import balanced_accuracy_score, f1_score
test_balacc = balanced_accuracy_score(labels, preds)
test_f1m    = f1_score(labels, preds, average="macro")
print(f"  TEST ACCURACY     : {test_acc*100:.2f}%   (enga#a con clases desbalanceadas)")
print(f"  BALANCED ACCURACY : {test_balacc*100:.2f}%   <- metrica principal")
print(f"  F1 MACRO          : {test_f1m:.4f}        <- metrica principal")
print(f"{'='*50}\n")
print(classification_report(labels, preds, target_names=CLASSES, digits=3))

NameError: name 'test_loader' is not defined

In [ ]:
# ── CELDA 21: Matriz de confusión ─────────────────────────────────────────────
cm = confusion_matrix(labels, preds, normalize="true")

fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor("#0f0f23")
ax.set_facecolor("#1a1a2e")

sns.heatmap(
    cm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=CLASSES, yticklabels=CLASSES,
    ax=ax, linewidths=0.5, linecolor="#333355",
    annot_kws={"size": 14, "color": "white"}
)
ax.set_title("Matriz de Confusión (normalizada)", color="white", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Predicción", color="gray", fontsize=11)
ax.set_ylabel("Real", color="gray", fontsize=11)
ax.tick_params(colors="gray")

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(colors="gray")

plt.tight_layout()
plt.show()

In [ ]:
# ── CELDA 22: Curvas ROC por clase ────────────────────────────────────────────
labels_bin = label_binarize(labels, classes=[0, 1, 2])
palette    = ["#e74c3c", "#f39c12", "#2ecc71"]

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor("#0f0f23")
ax.set_facecolor("#1a1a2e")

for i, (cls, color) in enumerate(zip(CLASSES, palette)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], probs[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{cls} (AUC = {roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "w--", linewidth=0.8, alpha=0.4, label="Aleatorio")
ax.set_xlabel("False Positive Rate", color="gray")
ax.set_ylabel("True Positive Rate", color="gray")
ax.set_title("Curvas ROC por clase", color="white", fontsize=13, fontweight="bold")
ax.legend(facecolor="#1a1a2e", labelcolor="white")
ax.tick_params(colors="gray")
ax.spines[:].set_color("#333355")
ax.grid(True, alpha=0.12, color="white")

plt.tight_layout()
plt.show()

In [ ]:

# ── CELDA 23: Grad-CAM — ¿Qué parte del espectrograma mira la red? ────────────
# Implementación manual de Grad-CAM para la última capa convolucional

class GradCAM:
    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None

        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, img_tensor, class_idx):
        self.model.eval()
        output = self.model(img_tensor)
        self.model.zero_grad()
        output[0, class_idx].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = (weights * self.activations).sum(dim=1).squeeze()
        cam     = torch.clamp(cam, min=0)
        cam     = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy()


# Acceder a la última capa conv del modelo custom
# Selecciona la ultima capa convolucional segun la arquitectura del mejor modelo
if hasattr(model, "features"):            # CNN custom
    target_layer = [m for m in model.features.modules()
                    if isinstance(m, nn.Conv2d)][-1]
else:                                      # ResNet (model.model.layer4)
    target_layer = [m for m in model.modules()
                    if isinstance(m, nn.Conv2d)][-1]
grad_cam     = GradCAM(model, target_layer)

# Tomar 6 ejemplos del test set
sample_imgs, sample_labels = next(iter(DataLoader(test_ds, batch_size=6, shuffle=True)))
sample_imgs  = sample_imgs.to(DEVICE)
sample_preds = model(sample_imgs).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.patch.set_facecolor("#0f0f23")
fig.suptitle("Grad-CAM — Zonas del espectrograma que activan la decisión", color="white", fontsize=13, fontweight="bold")

unnorm = transforms.Normalize(mean=[-1, -1, -1], std=[2, 2, 2])

for col in range(6):
    img_t  = sample_imgs[col:col+1]
    pred   = sample_preds[col].item()
    true   = sample_labels[col].item()
    cam    = grad_cam.generate(img_t, pred)

    # Imagen original
    img_show = unnorm(sample_imgs[col]).cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    axes[0, col].imshow(img_show)
    axes[0, col].axis("off")
    color = "#2ecc71" if pred == true else "#e74c3c"
    axes[0, col].set_title(f"Real: {CLASSES[true]}\nPred: {CLASSES[pred]}", color=color, fontsize=8)

    # Grad-CAM superpuesto
    import cv2
    cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    heatmap     = plt.cm.jet(cam_resized)[:, :, :3]
    overlay     = 0.55 * img_show + 0.45 * heatmap
    axes[1, col].imshow(np.clip(overlay, 0, 1))
    axes[1, col].axis("off")
    axes[1, col].set_title("Grad-CAM", color="gray", fontsize=8)

for ax in axes.flat:
    ax.set_facecolor("#1a1a2e")

plt.tight_layout()
plt.show()

In [ ]:
# ── CELDA 24: Predicción en tiempo real sobre un nuevo ticker ─────────────────
import yfinance as yf
from scipy import signal as sig

def predecir_ticker(ticker, window_days=60):
    """Descarga los últimos datos y predice la dirección del próximo movimiento."""
    df_new  = yf.download(ticker, period="6mo", interval="1d", progress=False)
    close   = df_new["Close"].squeeze().values
    returns = np.diff(np.log(close))
    ret_n   = (returns - returns.mean()) / (returns.std() + 1e-8)

    if len(ret_n) < window_days:
        print(f"⚠️  No hay suficientes datos para {ticker}")
        return

    window = ret_n[-window_days:]
    f_w, t_w, S_w = sig.spectrogram(
        window, fs=1.0,
        window=sig.windows.hann(min(32, window_days)),
        noverlap=min(28, window_days - 1),
        nfft=64, scaling="density"
    )
    img_data = 10 * np.log10(S_w + 1e-12)

    # Convertir a tensor
    fig_tmp, ax_tmp = plt.subplots(figsize=(1, 1), dpi=64)
    ax_tmp.pcolormesh(t_w, f_w, img_data, cmap="magma", shading="gouraud")
    ax_tmp.axis("off")
    fig_tmp.subplots_adjust(left=0, right=1, top=1, bottom=0)
    import tempfile
    tmp_path = os.path.join(tempfile.gettempdir(), f"{ticker}_latest.png")
    fig_tmp.savefig(tmp_path, dpi=64, bbox_inches="tight", pad_inches=0)
    plt.close(fig_tmp)

    from PIL import Image
    img   = Image.open(tmp_path).convert("RGB")
    img_t = eval_tf(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(img_t)
        probs_ = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred   = logits.argmax(dim=1).item()

    print(f"\n{'='*45}")
    print(f"  🎯 Ticker        : {ticker}")
    print(f"  📅 Ventana       : últimos {window_days} días")
    print(f"  🔮 Predicción    : {CLASSES[pred]}")
    print(f"{'─'*45}")
    for cls, p in zip(CLASSES, probs_):
        bar = "█" * int(p * 30)
        print(f"  {cls:5s}: {bar:<30} {p*100:5.1f}%")
    print(f"{'='*45}")

    # Mostrar el espectrograma que se usó
    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))
    fig2.patch.set_facecolor("#0f0f23")
    axes2[0].plot(df_new["Close"].squeeze().values[-window_days:], color="#2ecc71", linewidth=1.5)
    axes2[0].set_title(f"{ticker} — últimos {window_days} días", color="white")
    axes2[0].set_facecolor("#1a1a2e")
    axes2[0].tick_params(colors="gray")
    axes2[0].spines[:].set_color("#333355")

    axes2[1].pcolormesh(t_w, f_w, img_data, cmap="magma", shading="gouraud")
    axes2[1].set_title(f"Espectrograma  →  Predicción: {CLASSES[pred]}", color="white")
    axes2[1].set_facecolor("#1a1a2e")
    axes2[1].tick_params(colors="gray")
    axes2[1].spines[:].set_color("#333355")

    plt.tight_layout()
    plt.show()


# Probar con distintos activos
predecir_ticker("BTC-USD")
predecir_ticker("AAPL")